In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df = pd.read_csv(f"{path}/Q1_data.csv")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
df["Delivery_Time"].hist(bins=30, edgecolor='black')
plt.title("Target Distribution Delivery_Time")
plt.xlabel("Delivery_Time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()


In [ ]:
# Task 1: Write your code here:
print(df.shape)
df=df.drop("Order_ID",axis=1)
print(df.shape)


In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values =(df.isnull().sum()/ len(df)) * 100
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


print(f"Before: {df.shape}")


for x in ['Courier_Experience_yrs','Delivery_Time']:
  df[x]=df[x].fillna(df[x].mean())
for x in ['Weather', 'Traffic_Level', 'Time_of_Day']:
  df[x]=df[x].fillna(df[x].mode()[0])

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)



In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,accuracy_score
from sklearn.linear_model import Ridge, Lasso


models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)


all_results = {}

for name in models:
  all_results[name] = {'mae': [], 'acc': []}


for fold_idx, (train_index, test_index) in enumerate(kf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  ## Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)

    # Calculate evaluation metrics
    all_results[model_name]["mae"].append(mae)


for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MaE:  {np.mean(all_results[model_name]['mae']):.4f}")



In [ ]:
# Task 1: Write your code here:

coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: